In [4]:
import numpy as np

x = np.array([1, 2])
W1 = np.array([[1, 2], [-1, 1]])
b1 = np.zeros(2)
w2 = np.array([2, 1])
b2 = 1

def forward(x):
    z1 = x @ W1 + b1
    h = np.maximum(0, z1)
    return z1, h, h @ w2 + b2,


z1, h, y_hat = forward(x)
print(z1, h, y_hat)

[-1.  4.] [0. 4.] 5.0


In [5]:
def mse(y, p):
    return np.mean((p-y)**2)

def bce(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print(mse(np.array([2]), np.array([5])))
for p in (0.9, 0.5, 0.1):
    print(p, round(bce(1, p), 4))

9.0
0.9 0.1054
0.5 0.6931
0.1 2.3026


In [7]:
# 1
print(mse(8, 6))
# 2.
print(mse(np.array([8, 4, 6]), np.array([5, 4, 9])))

# 3.
print(bce(1, 0.8))
print(bce(0, 0.8))

# 4.
print(bce(1, 0.01))

4.0
6.0
0.2231435513142097
1.6094379124341005
4.605170185988091


### Back Propagation

In [8]:
# backward propagation
def backward(x, y):
    z1, h, y_hat = forward(x)
    d = 2 * (y_hat - y)
    dw2 = h * d
    dh = w2 * d
    dz1 = dh * (z1 > 0)
    dW1 = np.outer(x, dz1)
    return dW1, dz1, dw2, d

y = 2
dW1, db1, dw2, db2 = backward(x, y)
print(dW1, db1, dw2, db2)

[[ 0.  6.]
 [ 0. 12.]] [0. 6.] [ 0. 24.] 6.0


## MLP

In [19]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))

class MLP:
  def __init__(self, n_in, n_h, seed=0):
    r = np.random.default_rng(seed)
    self.W1 = r.normal(0, 1, (n_in, n_h))
    self.W2 = r.normal(0, 1, (n_h, 1))
    self.b1, self.b2 = np.zeros(n_h), np.zeros(1)

  def forward(self, X):
    self.h = sigmoid(X @ self.W1 + self.b1)
    self.p = sigmoid(self.h @ self.W2 + self.b2)
    return self.p

  def step(self, X, y, lr):
    dz2 = (self.p - y) / len(X)
    dz1 = dz2 @ self.W2.T * self.h * (1 - self.h)
    self.W2 -= lr * self.h.T @ dz2
    self.b2 -= lr * dz2.sum(0)
    self.W1 -= lr * X.T @ dz1
    self.b1 -= lr * dz1.sum(0)

## XOR oqitamiz

In [23]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

net = MLP(2, 4, seed = 1)
for ep in range(5001):
  p = net.forward(X)
  loss = -np.mean(y * np.log(p) + ( 1 - y) * np.log(1 - p))
  if ep in (0, 500, 1000, 5000):
    print(ep, round(loss, 4))
  net.step(X, y, lr=1.0)


print((net.forward(X) > 0.5).astype(int).ravel())

0 0.7452
500 0.0714
1000 0.014
5000 0.0016
[0 1 1 0]


## Working with PyTorch

In [68]:
import torch

x = torch.tensor([1., 2.])
W1 = torch.tensor([[1., 2.], [-1., 1.]], requires_grad=True)
b1 = torch.zeros(2, requires_grad=True)
w2 = torch.tensor([2., 1.], requires_grad=True)
b2 = torch.tensor(1., requires_grad=True)

y_hat = torch.relu(x @ W1 + b1) @ w2 + b2
loss = (y_hat - 2) ** 2
loss.backward()

print(W1.grad.numpy())
print(b1.grad.numpy())
print(w2.grad.numpy(), b2.grad.item())

[[ 0.  6.]
 [ 0. 12.]]
[0. 6.]
[ 0. 24.] 6.0


## Working with penguins

In [70]:
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

belgilar = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

df = sns.load_dataset('penguins').dropna(subset=belgilar)
X = df[belgilar].values
y = df['species'].astype('category').cat.codes.values
print(df.shape, np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(len(X_train), len(X_test), len(y_train), len(y_test))

(342, 7) [151  68 123]
239 103 239 103


## MLP with torch

In [72]:
import torch, torch.nn as nn

def fit_eval(X_train, y_train, X_test, y_test, scale=True, lr=0.1, hid=8, log=False):

    if scale:
        sc = StandardScaler().fit(X_train)
        X_train, X_test = sc.transform(X_train), sc.transform(X_test)

    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))
    opt = torch.optim.SGD(model.parameters(), lr=lr)


    for ep in range(201):
        loss = nn.functional.cross_entropy(model(X_train), y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if log and ep % 50 == 0:
            print(ep, loss.item())

    acc = (model(X_test).argmax(1) == y_test).float().mean()
    return acc.item()

print(fit_eval(X_train, y_train, X_test, y_test, log=True))

0 1.1499919891357422
50 0.2703212797641754
100 0.14031265676021576
150 0.09182716906070709
200 0.06672974675893784
0.9902912378311157
